# How to stream events from within a tool

If your LangGraph graph needs to use tools that call LLMs (or any other LangChain `Runnable` objects -- other graphs, LCEL chains, retrievers, etc.), you might want to stream events from the underlying `Runnable`. This guide shows how you can do that.

## Setup

In [ ]:
# DEPENDENCY: pip install --quiet -U langchain langchain-openai langchain-community langchain-text-splitters langgraph langgraph-prebuilt langgraph-checkpoint-sqlite langsmith ddgs
# (packages should be pre-installed in venv before running this script)

In [ ]:
import getpass
import os
import warnings

from openai import OpenAI

# Suppress deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langchain")

# Set up your OpenAI client
if not os.getenv("OPENAI_API_KEY"):
    secret_key = os.environ.get("OPENAI_API_KEY")
    os.environ["OPENAI_API_KEY"] = secret_key

## Define graph and tools

We'll use a prebuilt ReAct agent for this guide

In [3]:
from langchain_core.callbacks import Callbacks
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

<div class="admonition warning">
    <p class="admonition-title">ASYNC IN PYTHON<=3.10</p>
    <p>
Any Langchain RunnableLambda, a RunnableGenerator, or Tool that invokes other runnables and is running async in python<=3.10, will have to propagate callbacks to child objects manually. This is because LangChain cannot automatically propagate callbacks to child objects in this case.
    
This is a common reason why you may fail to see events being emitted from custom runnables or tools.
    </p>
</div>

In [4]:
@tool
async def get_items(place: str, callbacks: Callbacks) -> str:  # <--- Accept callbacks (Python <= 3.10)
    """Use this tool to look up which items are in the given place."""
    template = ChatPromptTemplate.from_messages(
        [
            (
                "human",
                "Can you tell me what kind of items i might find in the following place: '{place}'. "
                "List at least 3 such items separating them by a comma. And include a brief description of each item..",
            )
        ]
    )
    chain = template | llm.with_config(
        {
            "run_name": "Get Items LLM",
            "tags": ["tool_llm"],
            "callbacks": callbacks,  # <-- Propagate callbacks (Python <= 3.10)
        }
    )
    chunks = [chunk async for chunk in chain.astream({"place": place})]
    return "".join(chunk.content for chunk in chunks)

We're adding a custom tag (`tool_llm`) to our LLM runnable within the tool. This will allow us to filter events that we'll stream from the compiled graph (`agent`) Runnable below

In [5]:
llm = ChatOpenAI()
tools = [get_items]
agent = create_react_agent(llm, tools=tools)

## Stream events from the graph

In [ ]:
async for event in agent.astream_events({"messages": [("human", "what items are on the shelf?")]}, version="v2"):
    tags = event.get("tags", [])
    if event["event"] == "on_chat_model_stream" and "tool_llm" in tags:
        print(event["data"]["chunk"].content, end="", flush=True)

Let's inspect the last event to get the final list of messages from the agent

In [7]:
final_messages = event["data"]["output"]["messages"]

In [8]:
for message in final_messages:
    message.pretty_print()

================================ Human Message =================================

what items are on the shelf?
================================== Ai Message ==================================
Tool Calls:
  get_items (call_2YJ20GUM0fReVJam97fxouyL)
 Call ID: call_2YJ20GUM0fReVJam97fxouyL
  Args:
    place: shelf
================================= Tool Message =================================
Name: get_items

1. Books - Books are typically found on shelves, and they can range from novels to textbooks to cookbooks. They are made of paper and have information or stories written on their pages.

2. Picture frames - Picture frames are often placed on shelves to display photos or artwork. They come in various sizes, styles, and materials such as wood, metal, or plastic.

3. Decorative figurines - Decorative figurines are small sculptures or ornaments that can be displayed on shelves for aesthetic purposes. They can be made of various materials like porcelain, glass, or resin.
================

You can see that the content of the `ToolMessage` is the same as the output we streamed above